# 02 从三波段手算到 Painter Fig. 3

第一份 Notebook 表明雪粒径可以改变光谱。现在我想检验：**真实雪谱变了，解混模型却一直使用 200 μm 雪谱，雪比例会怎样偏移？** 先用三波段例子分清原始系数、阴影和归一化覆盖，再生成 14,746 条有明确真值的混合谱。

线性模型为 $R_c=\sum_iF_iR_{i,c}+E_c$，其中 $E_c=R_c-\widehat R_c$，$\mathrm{RMS}=\sqrt{\sum_cE_c^2/M}$。每次模型中只放一个候选雪谱和固定石英谱；多粒径方法轮流尝试候选，再选择 RMS 最小者。这样可将谱形选择与系数求解分开检查。

本页首行应打印 `三波段示例：观测 [0.62, 0.44, 0.20]`；批量计算应打印 `146 × 101 × 224 = 14,746 条光谱`。应看到原始 Fig. 3 对应的 RMS 面与纯雪曲线、观测—重建—残差图，以及 FCLS 噪声对照表。

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from snowmix import teaching as t
from snowmix.linear import mgs_operator, select_snow
t.configure_plots()
print("三波段示例：观测 [0.62, 0.44, 0.20]")

## 先看得见每个数的来历

取雪谱 `[0.9, 0.6, 0.2]`，裸地谱 `[0.4, 0.4, 0.4]`。它们是**教学合成谱**，三个数表示三个波段的无量纲反射率。设雪系数为 0.6、裸地系数为 0.2，剩余 0.2 为零反射率阴影；第一波段就是 $0.6\times0.9+0.2\times0.4=0.62$。

矩阵 `A` 的每列是一个端元，形状为 `(3 bands, 2 components)`；列向量观测满足 $y=Af$。先生成观测，再让求解器看不到真值地恢复系数。原文模式的 MGS 给出无约束最小二乘解；这里和 NumPy 的最小二乘交叉检查。

In [ ]:
snow_small = np.array([.9, .6, .2])
bare_small = np.array([.4, .4, .4])
A = np.column_stack((snow_small, bare_small))
true_raw = np.array([.6, .2])
y = A @ true_raw
raw_mgs = mgs_operator(A) @ y
raw_numpy = np.linalg.lstsq(A, y, rcond=None)[0]
shade = 1 - raw_mgs.sum()
normalized_snow = raw_mgs[0] / raw_mgs.sum()
residual_small = y - A @ raw_mgs
np.testing.assert_allclose(y, [.62, .44, .20], atol=1e-14)
np.testing.assert_allclose(raw_mgs, true_raw, atol=1e-14)
np.testing.assert_allclose(raw_mgs, raw_numpy, atol=1e-14)
assert abs(shade - .2) < 1e-14 and abs(normalized_snow - .75) < 1e-14
print("端元矩阵 A：\n", A)
print("恢复原始系数：", raw_mgs.round(6))
print(f"阴影 {shade:.2f}；阴影归一化雪比例 {normalized_snow:.2f}；RMS {np.sqrt(np.mean(residual_small**2)):.2e}")

恢复的 0.6 表示整个像元中雪的线性系数；去除阴影补量后，雪占被照亮组分的 $0.6/(0.6+0.2)=0.75$。这两个量回答不同问题，所以后面把原始系数用于重建，把阴影归一化比例用于覆盖对照。

若用归一化的 `[0.75, 0.25]` 直接重建，阴影会被消掉，亮度随之改变。可以手算第一波段：0.775 已不等于观测 0.62。逐波段残差 $E_c$ 正是检查这种错误最直接的量。

## 进入完整的半径—比例网格

为隔离雪谱失配，只保留雪和石英两种地物，真值不含阴影。半径 50–1500 μm 共 146 个，比例 0–1 共 101 个；每个组合是一条 224 波段观测。将物理雪谱和 USGS 石英谱卷积到同一组 AVIRIS 中心/FWHM，可以避免把波段差异误当作粒径效应。这里沿用原文流程的 0°模拟设定；课程 60°场景已在上一页展示。

`observation` 的第一轴按半径分组，每组 101 个比例。求解后可以还原为 `(146, 101, 224)`，从而把 RMS 画成半径—比例平面。改变这一小节以前，先运行后面的固定基线断言。

In [ ]:
inputs = t.synthetic_inputs(angle_deg=0, ice_option=1)
assert inputs["snow"].shape == (146, 224)
experiment = t.run_fig3(inputs)
assert experiment["observation"].shape == (14746, 224)
print("重新生成并解混：146 × 101 × 224 = 14,746 条光谱")

## 2 固定端元与多粒径候选

左图的误差使用固定 200 μm 雪谱；右图只保留真实雪比例为 1 的样本。多粒径模型每次放入一个雪谱，并选择 RMS 最低的候选。图中的越界系数用于解释最小二乘行为，输出未裁剪。

In [ ]:
figure = t.plot_fig3(experiment)
display(figure)
plt.close(figure)
metrics = t.fig3_metrics(experiment)
display(metrics.round(6))

### 数值检查与指标对象

表中的全网格 bias 是 14,746 个案例的平均误差；两个纯雪端点 bias 分别针对 50 μm 和 1500 μm。它们描述的统计对象不同。下面用匹配端元和已知真值检查实现。

In [ ]:
matching = experiment["truth_radius_um"] == 200
assert np.max(experiment["fixed"].rms[matching]) < 1e-8
assert np.max(experiment["multi"].rms) < 1e-8
np.testing.assert_allclose(experiment["multi"].normalized[:, 0], experiment["truth_fraction"], atol=1e-10)
with_snow = experiment["truth_fraction"] >= .1
np.testing.assert_array_equal(experiment["multi"].radius_um[with_snow], experiment["truth_radius_um"][with_snow])
print(f"多粒径最大 RMS = {np.max(experiment['multi'].rms):.3e}；已知比例与有雪半径检查通过。")

## 3 逐波段误差项

选择一个 500 μm、真实雪比例 0.6 的合成像元。观测、重建、残差共用相同波段；残差谱揭示了 RMS 平均值背后的结构。

In [ ]:
pixel = np.flatnonzero((experiment["truth_radius_um"] == 500) & np.isclose(experiment["truth_fraction"], .6))[0]
fig, axes = plt.subplots(2, 1, figsize=(9, 5.6), sharex=True)
w = inputs["wavelength_um"]
axes[0].plot(w, experiment["observation"][pixel], color="black", label="Observed")
for key, color in zip(["fixed", "multi"], t.COLORS):
    result = experiment[key]
    np.testing.assert_allclose(result.residual, experiment["observation"] - result.modeled, atol=1e-12)
    np.testing.assert_allclose(result.rms, np.sqrt(np.mean(result.residual**2, axis=1)), atol=1e-12)
    axes[0].plot(w, result.modeled[pixel], color=color, label=key)
    axes[1].plot(w, result.residual[pixel]*100, color=color, label=key)
axes[0].set_ylabel(t.label("反射率", "Reflectance")); axes[0].legend()
axes[1].set(xlabel=t.label("波长 / μm", "Wavelength / μm"), ylabel=t.label("E_c / 反射率百分点", "E_c / reflectance pp"))
axes[1].axhline(0, color=".4", lw=.8); axes[1].legend()
fig.tight_layout(); display(fig); plt.close(fig)

## 4 FCLS 与噪声示例

FCLS 要求非阴影系数非负、总和不超过 1，剩余比例为阴影。下例使用三种半径、两种雪比例和两种阴影，加入标准差 0.005 的相同噪声，比较无约束与 FCLS。随机种子固定为 1998。

In [ ]:
rng = np.random.default_rng(1998)
observations, truth = [], []
for radius in [50, 500, 1000]:
    spectrum = inputs["snow"][np.flatnonzero(inputs["radii_um"] == radius)[0]]
    for fraction in [.2, .6]:
        for shade in [0, .25]:
            observations.append((1-shade)*(fraction*spectrum + (1-fraction)*inputs["quartz"]))
            truth.append(fraction)
observations = np.asarray(observations) + rng.normal(0, .005, (12, 224))
rows = []
for solver in ["original", "fcls"]:
    result = select_snow(observations, inputs["snow"], inputs["radii_um"], inputs["quartz"], solver=solver)
    rows.append({"solver": solver, **t.difference_metrics(truth, result.normalized[:, 0]), "spectral_rms_mean": result.rms.mean()})
    if solver == "fcls":
        assert result.fractions.min() >= -1e-10
        assert result.fractions.sum(axis=1).max() <= 1 + 1e-10
display(pd.DataFrame(rows).round(6))

## 把结果用于下一步判断

低 RMS 表示光谱拟合接近观测。这里因为知道合成真值，还能单独检查雪比例与半径；实景要借助额外参考。对照表中的 FCLS 限制了比例范围，但约束本身不会补回遗漏的光谱特征。

练习可另建代码 cell：把 `t.synthetic_inputs(angle_deg=0)` 改为 60°，保留雪谱、石英和观测的共同波段；再比较固定 200 μm 的误差曲面。说明角度改变了哪个输入、哪些统计对象保持不变。不要改动上面的基线断言来适配练习结果。

下一步：[03 先设计验证，再解读真实结果](03_aviris_modis.ipynb)。